# LLM Gateway: reproducible GPU benchmark (Kaggle)

This notebook runs the **real** stack on a free Kaggle GPU and measures it:

1. Installs Ollama, Redis and the gateway, and pulls the same models used locally
   (`llama3.2`, `llama3.2:1b`).
2. Runs `scripts/bench.py` **direct to Ollama (baseline)** and **through the gateway** with
   the same workloads: gateway overhead, a realistic load test, the cache effect and
   streaming TTFT.
3. Runs `scripts/failover.py`: two Ollama nodes, one killed mid-run, measuring availability.
4. Saves everything to `/kaggle/working/results/` (download it from the **Output** tab).

**Before you run it:** turn on Settings → Accelerator → **GPU T4 x2** (or P100) and
Settings → **Internet on** (this needs a phone-verified Kaggle account). Then use
**Run All**. It takes about 15–25 minutes.

**Getting the code in:** set `REPO_URL` below if the GitHub repo is public. If it's
private, upload a zip of the repo as a Kaggle Dataset and attach it. The notebook finds
it under `/kaggle/input/`. Never put a GitHub token in a notebook.

In [ ]:
import json
import os
import secrets
import shutil
import signal
import subprocess
import sys
import threading
import time
import urllib.request
from pathlib import Path

REPO_URL = "https://github.com/GunjanPR/01-llm-gateway.git"  # works once the repo is public
WORK = Path("/kaggle/working")
REPO = WORK / "01-llm-gateway"
RESULTS = WORK / "results"
MODELS = ["llama3.2", "llama3.2:1b"]
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. Get the code

In [ ]:
def find_uploaded_repo() -> Path | None:
    for pyproject in Path("/kaggle/input").glob("**/pyproject.toml"):
        if (pyproject.parent / "src" / "llm_gateway").is_dir():
            return pyproject.parent
    return None


if not REPO.exists():
    uploaded = find_uploaded_repo()
    if uploaded:
        shutil.copytree(uploaded, REPO)
        print("using uploaded dataset:", uploaded)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
print(
    subprocess.run(
        ["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True
    ).stdout
    or "(not a git checkout)"
)

## 2. Install the gateway, Redis and Ollama

The project targets Python 3.11. Kaggle's Python version can differ, so the install skips
the version pin. That doesn't change what the benchmark measures.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python", str(REPO)],
    check=True,
)
subprocess.run(
    "apt-get -qq update && apt-get -qq install -y redis-server zstd > /dev/null",
    shell=True,
    check=True,
)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh > /dev/null", shell=True, check=True)
print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout)

## 3. Start Redis and Ollama, pull the models

In [ ]:
def wait_for(url: str, timeout: float = 120) -> None:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url, timeout=2)
            return
        except OSError:
            time.sleep(1)
    raise TimeoutError(url)


def start_ollama(port: int) -> subprocess.Popen:
    env = {**os.environ, "OLLAMA_HOST": f"127.0.0.1:{port}", "OLLAMA_KEEP_ALIVE": "1h"}
    log = open(RESULTS / f"ollama-{port}.log", "w")  # noqa: SIM115 - lives as long as the server
    proc = subprocess.Popen(
        ["ollama", "serve"], env=env, stdout=log, stderr=log, start_new_session=True
    )
    wait_for(f"http://127.0.0.1:{port}/api/version")
    return proc


subprocess.run(["redis-server", "--daemonize", "yes"], check=True)
ollama_main = start_ollama(11434)
for model in MODELS:
    subprocess.run(["ollama", "pull", model], check=True, stdout=subprocess.DEVNULL)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 4. Record the hardware

In [ ]:
gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,power.limit,clocks.max.sm,driver_version",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
).stdout.strip()
env_info = {"gpu": gpu.splitlines(), "python": sys.version.split()[0], "platform": "kaggle"}
(RESULTS / "environment.json").write_text(json.dumps(env_info, indent=2))
print(json.dumps(env_info, indent=2))

## 5. Start the gateway

It gets a throwaway API key generated here. Rate limiting is off (`RATE_LIMIT_RPM=0`) so
the limiter doesn't reject benchmark traffic.

In [ ]:
API_KEY = secrets.token_urlsafe(24)
gateway: subprocess.Popen | None = None


def start_gateway(ollama_urls: str) -> subprocess.Popen:
    global gateway
    if gateway:
        gateway.terminate()
        gateway.wait()
    env = {
        **os.environ,
        "GATEWAY_API_KEYS": API_KEY,
        "REDIS_URL": "redis://127.0.0.1:6379/0",
        "OLLAMA_BASE_URL": ollama_urls,
        "OLLAMA_MODELS": ",".join(MODELS),
        "RATE_LIMIT_RPM": "0",
        "LOG_LEVEL": "warning",
    }
    log = open(RESULTS / "gateway.log", "a")  # noqa: SIM115 - lives as long as the server
    gateway = subprocess.Popen(
        [sys.executable, "-m", "llm_gateway", "--port", "8000"],
        env=env,
        cwd=REPO,
        stdout=log,
        stderr=log,
    )
    wait_for("http://127.0.0.1:8000/health")
    return gateway


start_gateway("http://127.0.0.1:11434")
print(urllib.request.urlopen("http://127.0.0.1:8000/health").read().decode())

## 6. Benchmarks: direct to Ollama (baseline) vs through the gateway

In [ ]:
BENCH_OUT = RESULTS / "bench.jsonl"
DIRECT = ["--base-url", "http://127.0.0.1:11434/v1"]
GATEWAY = ["--base-url", "http://127.0.0.1:8000/v1", "--api-key", API_KEY]


def bench(label: str, target: list[str], *args: str) -> None:
    cmd = [
        sys.executable,
        "scripts/bench.py",
        "--label",
        label,
        "--out",
        str(BENCH_OUT),
        *target,
        *args,
    ]
    subprocess.run(cmd, cwd=REPO, check=True, stdout=subprocess.DEVNULL)
    print("done:", label)


# Gateway overhead: 1-token completions, rounds alternated so drift hits both sides equally.
for _ in range(3):
    common = ["--requests", "100", "--concurrency", "1", "--max-tokens", "1", "--warmup", "5"]
    bench("overhead-direct", DIRECT, *common)
    bench("overhead-gateway", GATEWAY, *common, "--no-cache")

# Realistic load: 64-token answers, 4 concurrent clients.
load = ["--requests", "80", "--concurrency", "4", "--max-tokens", "64"]
bench("load-direct", DIRECT, *load)
bench("load-gateway", GATEWAY, *load, "--no-cache")

# Cache effect: half of the prompts repeat an earlier one.
bench("cache-gateway", GATEWAY, *load, "--repeat-ratio", "0.5", "--seed", "11")

# Streaming time-to-first-token.
stream = ["--requests", "40", "--concurrency", "1", "--max-tokens", "64", "--stream"]
bench("stream-direct", DIRECT, *stream)
bench("stream-gateway", GATEWAY, *stream, "--no-cache")

## 7. Results

In [ ]:
import pandas as pd

rows = [json.loads(line) for line in BENCH_OUT.read_text().splitlines()]
df = pd.DataFrame(rows)
cols = [
    "label",
    "requests",
    "errors",
    "latency_p50_s",
    "latency_p95_s",
    "ttft_p50_s",
    "throughput_rps",
    "completion_tokens_per_s",
    "cache_hits",
]
summary = df[cols].groupby("label", sort=False).mean(numeric_only=True).round(4)
summary.to_csv(RESULTS / "summary.csv")
summary

In [ ]:
def p50(label: str, col: str = "latency_p50_s") -> float:
    return float(df.loc[df.label == label, col].median())


overhead_ms = (p50("overhead-gateway") - p50("overhead-direct")) * 1000
load_overhead = p50("load-gateway") / p50("load-direct") - 1
cache_gain = (
    float(df.loc[df.label == "cache-gateway", "throughput_rps"].iloc[0])
    / float(df.loc[df.label == "load-gateway", "throughput_rps"].iloc[0])
    - 1
)
ttft_ms = (p50("stream-gateway", "ttft_p50_s") - p50("stream-direct", "ttft_p50_s")) * 1000
headline = {
    "gpu": env_info["gpu"],
    "gateway_overhead_p50_ms_1_token": round(overhead_ms, 1),
    "gateway_overhead_p50_pct_64_tokens": round(load_overhead * 100, 2),
    "cache_throughput_gain_pct_50pct_repeats": round(cache_gain * 100, 1),
    "ttft_added_p50_ms": round(ttft_ms, 1),
    "direct_completion_tokens_per_s_4_clients": round(
        float(df.loc[df.label == "load-direct", "completion_tokens_per_s"].iloc[0]), 1
    ),
}
(RESULTS / "headline.json").write_text(json.dumps(headline, indent=2))
print(json.dumps(headline, indent=2))

## 8. Failover: kill a backend under load

A second Ollama node starts on port 11435 and goes first in the gateway's backend list,
so it serves traffic. After 20 s of a 60 s run, the notebook kills it. The gateway should
retry, open that node's circuit breaker, and send the rest of the traffic to the main node.

In [ ]:
node2 = start_ollama(11435)
start_gateway("http://127.0.0.1:11435,http://127.0.0.1:11434")


def kill_node2() -> None:
    os.killpg(node2.pid, signal.SIGKILL)
    print("killed ollama-2 at", time.strftime("%X"))


threading.Timer(20, kill_node2).start()
subprocess.run(
    [
        sys.executable,
        "scripts/failover.py",
        "--api-key",
        API_KEY,
        "--duration",
        "60",
        "--concurrency",
        "2",
        "--out",
        str(RESULTS / "failover.jsonl"),
    ],
    cwd=REPO,
    check=True,
)

## 9. Prometheus metrics snapshot and cleanup

In [ ]:
metrics = urllib.request.urlopen("http://127.0.0.1:8000/metrics").read().decode()
(RESULTS / "metrics.prom").write_text(metrics)
print(
    "\n".join(
        line
        for line in metrics.splitlines()
        if line.startswith(("llmgw_requests_total", "llmgw_backend_errors_total"))
    )
)
for proc in (gateway, ollama_main):
    if proc:
        proc.terminate()
print("results saved to", RESULTS, sorted(p.name for p in RESULTS.iterdir()))